Notebook prepared by Henrique Lopes Cardoso (hlc@fe.up.pt), based on [minbpe](https://github.com/karpathy/minbpe) by Andrej Karpathy. He also has a very nice introduction to GPT tokenizers [here](https://www.youtube.com/watch?v=zduSFxRajkE).

# BYTE PAIR ENCODING

Byte Pair Encoding is a well-known tokenization algorithm that proceeds by making merges of the most frequent adjacent token pairs. It starts with a vocabulary including every single character (in our case, we will use UTF-8 encoding), and then adds new tokens to the vocabulary by merging adjacent ones, as found in the training data.

In this notebook, you will implement the Byte Pair Encoding algorithm based on three main functions:

- `def train(text, vocab_size, verbose=False)`: the function where you will train the BPE tokenizer on some data, and that will give us as an output the *merges* that were made and the trained *vocabulary*.
- `def encode(text)`: the function that, given some text, tokenizes it based on the merges obtained from the training step of the tokenizer.
- `def decode(ids)`: the function that, given some sequence of tokens, reconstructs the original text.

> Note that GPT tokenizers, such as GPT-4's, includes additional complexity than what is covered in this notebook, including:
> - Splitting the text into chunks using a regex pattern before tokenization, meant to ensure that no merges will happen across category boundaries (letters, numbers, punctuation).
> - Including special tokens that are not part of the text and are specific to the functioning of the language model.

## Auxiliary functions

We'll start by defining two auxiliary functions that will be helpful later on.

The first of these functions is `merge`: given a list of integers `ids` (token identifiers) and a `pair`, it replaces every consecutive occurrence of the pair in the list with some other identifier `idx`.

In [1]:
def merge(ids, pair, idx):
    """
    In the list of integers ids, replace all consecutive occurrences of pair with the new integer token idx
    Example: ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4 -> [4, 3, 4]
    """
    newids = []
    # your code here
    
    i = 0
    while i < len(ids):
        if (i < len(ids) - 1) and ((ids[i], ids[i+1]) == pair):
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
            

    return newids

Make sure to test your `merge` function:

In [2]:
test1 = merge(ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4)
assert test1 == [4,3,4]

# other tests here
test2 = merge(ids=[4, 5, 7, 1, 3, 5, 8, 9, 4, 5, 1, 3, 2, 5, 4], pair=(4, 5), idx=-1)
assert test2 == [-1, 7, 1, 3, 5, 8, 9, -1, 1, 3, 2, 5, 4]

The second function, `get_pair_counts`, counts the number of times each consecutive pair occurs in a sequence of integers.

In [3]:
def get_pair_counts(ids, counts=None):
    """
    Given a list of integers ids, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    # your code here
    
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i+1])
        if pair not in counts:
            counts[pair] = 0
        counts[pair] += 1
    
    return counts

Make sure to test your `get_pair_counts` function:

In [4]:
test1 = get_pair_counts([1, 2, 3, 1, 2])
assert test1 == {(1, 2): 2, (2, 3): 1, (3, 1): 1}

# other tests here
test2 = get_pair_counts([1, 4, 5, 2, 2, 1, 4, 5, 1, 4])
assert test2 == {(1, 4): 3, (4, 5): 2, (5, 2): 1 , (2, 2): 1, (2, 1): 1, (5, 1): 1}

## Train

The `train` function will build the tokenizer. That is, given some training corpus and the desired vocabulary size, it will determine what merges to make based on the frequency of adjacent tokens. It will also provide us with the resulting vocabulary.

In [5]:
def train(text, vocab_size, verbose=False):
    assert vocab_size >= 256   # we should do at least one merge...
    
    vocab = {idx: bytes([idx]) for idx in range(256)} # int -> bytes
    num_merges = vocab_size - 256

    # input text preprocessing: encode text using UTF-8
    text_bytes = text.encode("utf-8") # raw bytes
    ids = list(text_bytes) # list of integers in range 0..255

    # iteratively merge the most common pairs to create new tokens
    merges = {} # (int, int) -> int
    for i in range(num_merges):
        
        # get the pair with the highest number of occurrences
        # a) count up the number of times every consecutive pair appears
        counts = get_pair_counts(ids)
        # b) find the pair with the highest count
        pair = max(counts, key=counts.get)
        
        # use the next available id for the new token
        idx = 256 + i
        
        # replace all occurrences of pair in ids with idx
        ids = merge(ids, pair, idx)
        
        # save the merge
        merges[pair] = idx   # merges
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]   # vocabulary
        
        # prints
        if verbose:
            print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {counts[pair]} occurrences")

    # return merges and vocab
    return merges, vocab

Let's train the tokenizer in some simple text:

In [6]:
text = "aaabdaaabac"
merges, vocab = train(text, 256 + 3, verbose=True)

merge 1/3: (97, 97) -> 256 (b'aa') had 4 occurrences
merge 2/3: (256, 97) -> 257 (b'aaa') had 2 occurrences
merge 3/3: (257, 98) -> 258 (b'aaab') had 2 occurrences


And let's see what merges were made and what has been added to the vocabulary:

In [7]:
merges

{(97, 97): 256, (256, 97): 257, (257, 98): 258}

In [8]:
for key, value in vocab.items():
    if key > 255:
        print(key, value)

256 b'aa'
257 b'aaa'
258 b'aaab'


Train the tokenizer in some longer text, for a vocabulary size of 512.

In [9]:
# your code here
f = open('data/taylorswift.txt')
text = f.read()
merges, vocab = train(text, 512, verbose=True)

merge 1/256: (101, 32) -> 256 (b'e ') had 2981 occurrences
merge 2/256: (44, 32) -> 257 (b', ') had 2961 occurrences
merge 3/256: (100, 32) -> 258 (b'd ') had 2617 occurrences
merge 4/256: (46, 32) -> 259 (b'. ') had 2560 occurrences
merge 5/256: (114, 32) -> 260 (b'r ') had 2428 occurrences
merge 6/256: (50, 48) -> 261 (b'20') had 2365 occurrences
merge 7/256: (115, 32) -> 262 (b's ') had 2053 occurrences
merge 8/256: (105, 110) -> 263 (b'in') had 2006 occurrences
merge 9/256: (111, 110) -> 264 (b'on') had 1815 occurrences
merge 10/256: (114, 105) -> 265 (b'ri') had 1805 occurrences
merge 11/256: (116, 32) -> 266 (b't ') had 1802 occurrences
merge 12/256: (116, 104) -> 267 (b'th') had 1737 occurrences
merge 13/256: (101, 258) -> 268 (b'ed ') had 1736 occurrences
merge 14/256: (257, 261) -> 269 (b', 20') had 1705 occurrences
merge 15/256: (97, 110) -> 270 (b'an') had 1487 occurrences
merge 16/256: (97, 114) -> 271 (b'ar') had 1360 occurrences
merge 17/256: (101, 260) -> 272 (b'er ') ha

## Encode

The `encode` function will convert a text into a sequence of tokens, based on the merges imposed by the tokenizer.

In [10]:
def encode(text):
    # given a string text, return the token ids
    text_bytes = text.encode("utf-8") # raw bytes
    ids = list(text_bytes) # list of integers in range 0..255
    while len(ids) >= 2:
        # find the pair with the lowest merge index
        counts = get_pair_counts(ids)   # note that here we don't really care about the counts, we just need the observed pairs
        pair = min(counts.keys(), key=lambda p: merges.get(p, float("inf")))
        # subtle: if there are no more merges available, the key will
        # result in an inf for every single pair, and the min will be
        # just the first pair in the list, arbitrarily
        # we can detect this terminating case by a membership check
        if pair not in merges:
            break # nothing else can be merged anymore
        # otherwise let's merge the best pair (lowest merge index)
        idx = merges[pair]
        ids = merge(ids, pair, idx)
    
    return ids

Try encoding some piece of text. Compare the number of tokens obtained with the length of the text (or, more precisely, its UTF-8 encoding). This can be seen as a compression ratio (the original version of the [BPE algorithm](https://en.wikipedia.org/wiki/Byte_pair_encoding) focused on compression).
You can also compare the number of tokens with the number of words (e.g., as provided by NLTK's word tokenizer).

In [11]:
# your code here
text = "In nominae solenissima Praxis, Inphormaticus Engenhus cadeirum PLN passatum est"
ids = encode(text)
print(ids)

[427, 32, 110, 286, 263, 97, 256, 115, 407, 290, 367, 115, 447, 338, 80, 472, 120, 367, 257, 427, 112, 104, 291, 109, 388, 99, 117, 262, 69, 110, 103, 290, 104, 117, 262, 99, 396, 101, 105, 114, 353, 32, 80, 76, 78, 32, 112, 385, 115, 347, 353, 32, 310, 116]


Print individually the textual entry in the vocabulary corresponding to each of the first 25 tokens.

In [12]:
# your code here
for token_id in ids[:25]:
    token_bytes = vocab[token_id]
    token_text = token_bytes.decode("utf-8", errors="replace")
    print(token_text)
    

In
 
n
om
in
a
e 
s
ol
en
is
s
im
a 
P
ra
x
is
, 
In
p
h
or
m
ati


## Decode

The `decode` function will convert sequence of tokens back to text, based on the tokenizer's vocabulary.

In [13]:
def decode(ids):
    # given ids (list of integers), return Python string
    text_bytes = b"".join(vocab[idx] for idx in ids)
    text = text_bytes.decode("utf-8", errors="replace")
    return text

Decode back the text you used in the previous step. Check that you get back the original text.

In [14]:
# your code here
decode(ids)

'In nominae solenissima Praxis, Inphormaticus Engenhus cadeirum PLN passatum est'